# 模块概述

WtRiskMonFact 是 WonderTrader 风险管理工厂模块，负责提供各种风险监控算法，实现组合盘的风险管理和资金保护。主要包括：
- 风控模块工厂管理和创建
- 基于回撤控制的风险监控算法
- 日内和多日双重风控保护机制
- 独立线程异步风控检查
- 仓位控制和盈利保护

1. **工厂层**（WtRiskMonFact）：
   - 实现 IRiskMonitorFact 接口，提供风控模块的创建、删除和管理功能
   - 支持风控模块的枚举和查询功能
   - 实现风控模块的生命周期管理
   - 提供C接口函数，支持动态库加载

2. **风控监控器层**（WtSimpleRiskMon）：
   - **简单风控监控器**：实现基于回撤控制的风险监控功能
   - **日内回撤风控**：监控日内从最高点的回撤幅度，超过阈值时降低仓位
   - **多日回撤风控**：监控多日最大动态权益的回撤幅度，超过阈值时清仓
   - **盈利保护**：当盈利达到一定比例后，启用回撤保护机制
   - **定时检查**：按照设定的时间间隔定期检查风险状况
   - **仓位控制**：通过设置数量倍数（vol_scale）来控制整体仓位比例

3. **风控算法特点**：
   - **日内回撤计算**：rate = (maxBal - curBal) * 100 / (maxBal - predynbal)
   - **多日回撤计算**：rate = (maxBal - curBal) * 100 / maxBal
   - **时间窗口控制**：使用日内分钟数计算时间差，避免午盘休息时间影响风控判断
   - **双重保护机制**：同时支持日内和多日风控，提供双重保护

4. **架构特点**：
   - **多线程设计**：使用独立线程执行风控检查，不阻塞主交易流程
   - **可配置参数**：支持通过配置文件设置各种风控参数，灵活适应不同策略需求
   - **精确时间控制**：使用毫秒级时间戳和睡眠机制，精确控制检查间隔
   - **详细日志记录**：记录每次检查的详细情况，便于分析和调试

# 层次关系图
```mermaid
graph LR
    %% 样式定义
    classDef factoryClass fill:#e1f5ff,stroke:#01579b,stroke-width:3px,color:#000;
    classDef monitorClass fill:#fff3e0,stroke:#e65100,stroke-width:2px,color:#000;
    classDef interfaceClass fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;
    classDef contextClass fill:#e8f5e9,stroke:#1b5e20,stroke-width:2px,color:#000;

    %% 接口层
    subgraph Interfaces["接口层 - 抽象接口"]
        direction TB
        IRiskMonitorFact["IRiskMonitorFact<br/>风控模块工厂接口<br/>• 创建风控模块<br/>• 删除风控模块<br/>• 枚举风控模块<br/>• 获取工厂名称"]:::interfaceClass
        WtRiskMonitor["WtRiskMonitor<br/>风控模块基类<br/>• 初始化接口<br/>• 启动接口<br/>• 停止接口<br/>• 获取名称接口"]:::interfaceClass
        WtPortContext["WtPortContext<br/>组合上下文接口<br/>• 资金信息查询<br/>• 交易状态查询<br/>• 仓位控制接口<br/>• 日志记录接口<br/>• 时间转换接口"]:::contextClass
    end

    %% 工厂层
    subgraph Factory["工厂层 - 风控模块工厂"]
        direction TB
        WtRiskMonFact["WtRiskMonFact<br/>风控模块工厂<br/>• 创建风控模块<br/>• 删除风控模块<br/>• 枚举风控模块<br/>• C接口导出"]:::factoryClass
    end

    %% 风控监控器层
    subgraph Monitors["风控监控器层 - 风险监控实现"]
        direction TB
        WtSimpleRiskMon["WtSimpleRiskMon<br/>简单风控监控器<br/>• 日内回撤风控<br/>• 多日回撤风控<br/>• 盈利保护机制<br/>• 定时检查机制<br/>• 仓位控制<br/>• 独立线程执行"]:::monitorClass
    end

    %% 继承关系
    WtRiskMonFact -.->|"实现"| IRiskMonitorFact
    WtSimpleRiskMon -.->|"继承"| WtRiskMonitor

    %% 工厂创建关系
    WtRiskMonFact -->|"创建"| WtSimpleRiskMon

    %% 风控监控器使用组合上下文
    WtSimpleRiskMon -->|"使用"| WtPortContext
    WtSimpleRiskMon -.->|"初始化"| WtPortContext

    %% 应用样式
    class WtRiskMonFact factoryClass
    class WtSimpleRiskMon monitorClass
    class IRiskMonitorFact,WtRiskMonitor interfaceClass
    class WtPortContext contextClass
```